In [ ]:
%%capture
import os
import pandas as pd
from dj_notebook import activate
from pathlib import Path
env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
from intecomm_analytics.dataframes import get_df_main_1858
from edc_pdutils.dataframes import get_subject_visit, get_appointments
from edc_appointment.constants import SKIPPED_APPT, NEW_APPT, COMPLETE_APPT, INCOMPLETE_APPT, \
    IN_PROGRESS_APPT, ONTIME_APPT, MISSED_APPT, CANCELLED_APPT
from scipy.stats import ttest_ind
import numpy as np
from intecomm_analytics.utils import get_primary_cohorts_by_categorical_column
from intecomm_analytics.dataframes.get_location_update import get_location_update
from edc_pdutils.dataframes import get_crf
from edc_constants.constants import YES, NO
from edc_model_to_dataframe import read_frame_edc
from intecomm_prn.models import SubjectTransfer
from edc_constants.constants import OTHER
from great_tables import GT, html, loc, style
from intecomm_analytics.utils import get_primary_cohorts_for_continuous_var

from tabulate import tabulate


In [ ]:
df_main = get_df_main_1858(None)


In [ ]:
def get_attended(df):
    df["attended"] = "unknown"
    df.loc[df.appt_status.isin([NEW_APPT]), "attended"] = "not_reported"
    df.loc[df.appt_status.isin([CANCELLED_APPT]), "attended"] = "cancelled"
    df.loc[df.appt_status.isin([INCOMPLETE_APPT, IN_PROGRESS_APPT, COMPLETE_APPT]) & (df.appt_timing==MISSED_APPT), "attended"] = "missed"
    df.loc[df.appt_status.isin([INCOMPLETE_APPT, IN_PROGRESS_APPT, COMPLETE_APPT]) & (df.appt_timing==ONTIME_APPT), "attended"] = "attended"
    df.loc[df.appt_status.isin([SKIPPED_APPT]), "attended"] = "skipped"
    return df


In [ ]:
df_appointment = get_appointments()
df_appointment = df_appointment.merge(df_main[["subject_identifier", "assignment", "primary_cohort", "primary_cohort_str", "ncd", "hiv_only"]], on="subject_identifier", how="left")
df_appointment = get_attended(df_appointment)

In [ ]:
# Number of visits scheduled – approx. is fine
# Number of scheduled visits attended
tbl_dct = get_primary_cohorts_by_categorical_column(
    df_appointment[
        (df_appointment.visit_code_sequence==0) &
        ~(df_appointment.attended.isin(["skipped", "cancelled"]))
    ],
    "attended")
dftbl = pd.DataFrame(tbl_dct)
dftbl_scheduled = dftbl.copy()
dftbl_scheduled.loc[dftbl_scheduled.Statistics=="n", "Statistics"] = "Total number of scheduled visits"
dftbl_scheduled.loc[dftbl_scheduled.Statistics=="attended", "Statistics"] = " -scheduled visits atttended"
dftbl_scheduled.loc[dftbl_scheduled.Statistics=="missed", "Statistics"] = " -scheduled visits missed"
dftbl_scheduled.loc[dftbl_scheduled.Statistics=="not_reported", "Statistics"] = " -scheduled visits not reported"


In [ ]:
# Total number of unscheduled visits
tbl_dct = get_primary_cohorts_by_categorical_column(
    df_appointment[
        (df_appointment.visit_code_sequence!=0) &
        ~(df_appointment.attended.isin(["not_reported", "skipped", "cancelled"]))
    ], "attended")
dftbl = pd.DataFrame(tbl_dct)
dftbl.loc[dftbl.Statistics=="n", "Statistics"] = "Total number of unscheduled visits"
dftbl_unscheduled = dftbl[dftbl.Statistics=="Total number of unscheduled visits"].copy()
dftbl_unscheduled

In [ ]:
df1 = df_appointment[
        (df_appointment.visit_code_sequence!=0) &
        ~(df_appointment.attended.isin(["not_reported", "skipped", "cancelled"]))
    ].groupby(by=["subject_identifier"]).size().to_frame().reset_index()
df1["unscheduled"] = 1
df1 = df1.merge(df_main[["subject_identifier", "primary_cohort", "assignment", "primary_cohort_str", "ncd", "hiv_only"]], on="subject_identifier", how="left")
tbl_dct = get_primary_cohorts_by_categorical_column(
    df1, "unscheduled")
dftbl = pd.DataFrame(tbl_dct)
dftbl.loc[dftbl.Statistics=="n", "Statistics"] = "Number of patients who made one or more unscheduled visits"
dftbl_unscheduled_patients = dftbl[dftbl.Statistics=="Number of patients who made one or more unscheduled visits"].copy()
dftbl_unscheduled_patients

In [ ]:
# Total number subjects who attended an unscheduled visit
tbl_dct = get_primary_cohorts_by_categorical_column(
    df_appointment[
        (df_appointment.visit_code_sequence!=0) &
        ~(df_appointment.attended.isin(["not_reported", "skipped", "cancelled"]))
    ], "attended")
dftbl = pd.DataFrame(tbl_dct)
dftbl.loc[dftbl.Statistics=="n", "Statistics"] = "Total number of unscheduled visits"
dftbl_unscheduled = dftbl[dftbl.Statistics=="Total number of unscheduled visits"].copy()
dftbl_unscheduled

In [ ]:
# Total number of referrals to the next level of care (facility care for those in the community arm; hospital / tertiary care for those in the facility arm).
df_location_update = get_location_update(df_main)
df_location_update = df_location_update.merge(df_main[["subject_identifier", "primary_cohort", "primary_cohort_str", "ncd", "hiv_only"]], on="subject_identifier", how="left")
df_location_update = get_attended(df_location_update)
df_location_update = df_location_update[
        (df_location_update.visit_code<=1120.0) &
        ~(df_location_update.attended.isin(["not_reported", "skipped", "cancelled"]))
    ].copy().reset_index(drop=True)
tbl_dct = get_primary_cohorts_by_categorical_column(
    df_location_update, "direction")
dftbl = pd.DataFrame(tbl_dct)
dftbl2 = dftbl.copy()
dftbl2.loc[dftbl.Statistics=="n", "Statistics"] = "Community to facility for any reason"
dftbl2= dftbl2[dftbl2.Statistics=="Community to facility for any reason"].copy()


df1 = df_location_update.groupby(by=["subject_identifier"]).size().to_frame().reset_index()
df1["a->b"] = 1
df1 = df1.merge(df_main[["subject_identifier", "primary_cohort", "assignment", "primary_cohort_str", "ncd", "hiv_only"]], on="subject_identifier", how="left")
tbl_dct = get_primary_cohorts_by_categorical_column(
    df1, "a->b")
dftbl = pd.DataFrame(tbl_dct)
dftbl.loc[dftbl.Statistics=="n", "Statistics"] = "Number of patients from the community who attended the facility for any reason"
dftbl2a = dftbl[dftbl.Statistics=="Number of patients from the community who attended the facility for any reason"].copy()



In [ ]:
def get_location_update_reason(s):
    options = [
        "She goes to the village to find her husband and farming",
        "Patient wanted facility care because  of low payments for drugs",
        "She had gone for burial",
        "Patient didn't come due to  fear of stigma from a relative who is part of the group",
        "the participant was scheduled for a viral load test on this visit",
        "Integrated community clinic was not ready",
        "missed to attend community clinic, attended at facility",
        "missed community appointment, attended at facility",
        "Patient forgot to attend the community area",
        "PATIENT HAD FAMILY EMERGENCY AT THE DATE SCHEDULED FOR COMMUNITY VISIT",
        "Integrated community clinic is not ready",

    ]
    refill_options = [
        "To get her medication now need fingerprints and physical presence of patient at hospital",
        "Needs fingerprints to access her drugs",
        "patient got family emergency and want to travel, asked to attend facility for drug refill",
        "came for drug refills, completed eleventh month at the community",
        "Drug pick up",
        "Had an orthopedic clinic so she refilled drugs for HTN and DM",
        "For drug refills",
        "FOR DRUG REFILL",
        "Patient was having nhif insurance has to attend favility for finger print",
        "NHIF Patient attending facility",
        "Nhig patient attemdend facility",
        "He is using medication that are available through insurance",
        "Patient attended for routine appointment which doesn’t fall in community appointment",
        "He is using medication that are available through insurance only",
    ]
    care_options = [
        "Patient was referred back to facility because she has early sign of foot ulcer",
        "Missed her visit but also the foot ulce reoccur so she has to receive a treatment at hospital",
        "She was Sick during community clinic appointment.",
        "Patient condition is not well to attend for community services",
        "Lack of sleep and difficult in breathing",
        "Patient came to seek care for the illness",
        "Patient reffer herself to hospital for specialist care",
        "Patient visit for further management and collecting drugs"
        "Patient visit facility for further care because last seen patient got compication",
        "Patient was sick and so attended to the facility",
        "Patient felt unwell and attended facility",
        "Patient visit facility for further care because last seen patient got compication",
    ]
    if s["comments"] in options:
        return OTHER
    elif s["comments"] in care_options:
        return "Community to facility seeking care"
    elif s["comments"] in refill_options or "drug" in s["comments"] or "refill" in s["comments"]:
        return "drug_refill"
    return np.nan

df_location_update["location_update_reason"] = df_location_update.apply(get_location_update_reason, axis=1)


In [ ]:
tbl_dct = get_primary_cohorts_by_categorical_column(
    df_location_update[(df_location_update.direction=="a->b") & (df_location_update.location_update_reason=="Community to facility seeking care")], "location_update_reason")
dftbl = pd.DataFrame(tbl_dct)
dftbl3= dftbl[dftbl.Statistics=="n"].copy()
dftbl3.loc[dftbl3.Statistics=="n", "Statistics"] = "Community to facility seeking specialized care"

df1 = df_location_update[(df_location_update.direction=="a->b") & (df_location_update.location_update_reason=="Community to facility seeking care")].groupby(by=["subject_identifier"]).size().to_frame().reset_index()
df1["a->b"] = 1
df1 = df1.merge(df_main[["subject_identifier", "primary_cohort", "assignment", "primary_cohort_str", "ncd", "hiv_only"]], on="subject_identifier", how="left")
tbl_dct = get_primary_cohorts_by_categorical_column(df1, "a->b")
dftbl = pd.DataFrame(tbl_dct)
dftbl.loc[dftbl.Statistics=="n", "Statistics"] = "Number of community patients who sought specialized care at the facility"
dftbl3a = dftbl[dftbl.Statistics=="Number of community patients who sought specialized care at the facility"].copy()



In [ ]:
from intecomm_analytics.constants import DM_ALONE, HTN_ALONE, HTN_DM, HIV_ALONE, UNDEFINED

# primary_cohort
tbl_dct = get_primary_cohorts_by_categorical_column(df_main, "primary_cohort")
dftbl = pd.DataFrame(tbl_dct)
mapping = {DM_ALONE:"Diabetes alone", HTN_ALONE:"Hypertension alone", HTN_DM:"Diabetes and hypertension", HIV_ALONE:"HIV alone", UNDEFINED:"UNDEFINED", "n":"n"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl = dftbl[dftbl["Statistics"]!="UNDEFINED"]
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Diabetes alone", "Hypertension alone", "Diabetes and hypertension", "HIV alone"], ordered=True)
dftbl.sort_values(by=["Statistics"], ascending=True, inplace=True)
dftbl.replace("0 (0.0%)", "NA", inplace=True)
dfnum = dftbl.iloc[0:1]


In [ ]:

careseeka_df = get_crf("intecomm_subject.careseekinga",
                       subject_visit_model="intecomm_subject.subjectvisit")
careseeka_df = careseeka_df.merge(df_main[
                                      ["subject_identifier", "assignment", "primary_cohort",
                                       "primary_cohort_str", "ncd", "hiv_only"]],
                                  on=["subject_identifier"], how="left")
careseeka_df["care_visit_minutes"] = careseeka_df.care_visit_tdelta.dt.total_seconds() / 60

tbl_dct = get_primary_cohorts_for_continuous_var(
    careseeka_df, "care_visit_minutes", statistics=["median"])
dfcare_visit_minutes = pd.DataFrame(tbl_dct)
print(tabulate(dfcare_visit_minutes, showindex=False, headers="keys", tablefmt="simple_grid"))



In [ ]:
careseeka_df = get_crf("intecomm_subject.careseekinga",
                       subject_visit_model="intecomm_subject.subjectvisit")
careseeka_df = careseeka_df.merge(df_main[
                                      ["subject_identifier", "assignment", "primary_cohort",
                                       "primary_cohort_str", "ncd", "hiv_only"]],
                                  on=["subject_identifier"], how="left")
careseeka_df["travel_minutes"] = careseeka_df.travel_tdelta.dt.total_seconds() / 60

tbl_dct = get_primary_cohorts_for_continuous_var(
    careseeka_df, "travel_minutes", statistics=["median"])
dftravel_minutes = pd.DataFrame(tbl_dct)
print(tabulate(dftravel_minutes, showindex=False, headers="keys", tablefmt="simple_grid"))


In [ ]:
dftbl = pd.concat([dfnum, dftbl_scheduled, dftbl_unscheduled, dftbl_unscheduled_patients, dftbl2, dftbl2a, dftbl3, dftbl3a])
dftbl

In [ ]:
print(tabulate(dftbl, showindex=False, headers="keys", tablefmt="simple_grid"))


In [ ]:
df = dftbl.copy()
source_notes = ""
table = (
    GT(dftbl)
    .tab_header(title="Table 3").tab_spanner(
            label=html(
                "Participants with diabetes,<BR>hypertension, or both<br>"
                # f"(n={df.loc[0, ['Community Ncd', 'Facility Ncd']].sum()})"
            ),
            columns=[1, 2],
        ).tab_spanner(
            label=html(
                "Participants with<BR>HIV alone<BR>"
                # f"(n={df.loc[0, ['Community Hiv only', 'Facility Hiv only']].sum()})"
            ),
            columns=[3, 4],
        )
    # .cols_label(
    #         {
    #             "Community Ncd": html(
    #                 f"Community<BR>(n={df.loc[0, ['Community Ncd']].sum()})"
    #             ),
    #             "Facility Ncd": html(f"Facility<br>(n={df.loc[0, ['Facility Ncd']].sum()})"),
    #             "Community Hiv only": html(
    #                 f"Community<br>(n={df.loc[0, ['Community Hiv only']].sum()})"
    #             ),
    #             "Facility Hiv only": html(
    #                 f"Facility<br>(n={df.loc[0, ['Facility Hiv only']].sum()})"
    #             ),
    #         }
    #     )
     .cols_align(align="left", columns=[0])
     .cols_align(align="center", columns=[1, 2, 3, 4])
     .tab_stub(rowname_col="Statistics") #, groupname_col="group")
     .opt_stylize(style=3)
     .opt_row_striping(row_striping=False)
     .opt_vertical_padding(scale=1.2)
     .opt_horizontal_padding(scale=1.0)
     .tab_options(
        stub_background_color="white",
        row_group_border_bottom_style="hidden",
        row_group_padding=0.5,
        row_group_background_color="white",
        table_background_color="white",
        table_font_size=12,
    )
    .tab_style(
        style=[style.fill(color="white"), style.text(color="black")],
        locations=loc.body(columns=[1, 2, 3, 4], rows=list(range(0, len(df)))),
    )
    .tab_source_note(source_note=html(source_notes or ""))
    .tab_style(
        style=style.text(color="black", size="small"),
        locations=loc.footer())
)

table.show()

In [ ]:
from PIL import Image

# save as png
table.save(analysis_folder / "table3.png")
# export to PDF
image = Image.open(analysis_folder / "table3.png")
image = image.resize((image.width * 6, image.height * 6), Image.LANCZOS)
image.save(analysis_folder / "table3.pdf", "PDF", resolution=800, optimize=True, quality=95)

In [ ]:
complications_df = get_crf("intecomm_subject.ComplicationsFollowup", subject_visit_model="intecomm_subject.subjectvisit")

In [ ]:

def has_complications(s)->int:
    complication_count = 0
    if s["stroke"] == YES and s["stroke_date"] >= s["baseline_datetime"]:
        complication_count += 1
    if s["heart_attack"] == YES and s["heart_attack_date"] >= s["baseline_datetime"]:
        complication_count += 1
    if s["renal_disease"] == YES and s["renal_disease_date"] >= s["baseline_datetime"]:
        complication_count += 1
    if s["vision"] == YES and s["vision_date"] >= s["baseline_datetime"]:
        complication_count += 1
    if s["numbness"] == YES and s["numbness_date"] >= s["baseline_datetime"]:
        complication_count += 1
    if s["foot_ulcers"] == YES and s["foot_ulcers_date"] >= s["baseline_datetime"]:
        complication_count += 1
    if s["complications"] == YES:
        complication_count += 1
    return complication_count

complications_df["complications_count"] = complications_df.apply(has_complications, axis=1)



In [ ]:
df_appointment = df_appointment.merge(df_location_update[["subject_identifier", "visit_code", "direction"]], on=["subject_identifier", "visit_code"], how="left")
df_appointment = df_appointment.merge(complications_df[["subject_identifier", "visit_code", "complications_count"]], on=["subject_identifier", "visit_code"], how="left")

In [ ]:
df_appointment[(df_appointment.direction.notna()) | (df_appointment.complications_count.notna())][["subject_identifier", "visit_code", "direction", "complications_count"]]


In [ ]:
df_appointment[(df_appointment.direction.notna()) & (df_appointment.complications_count>0)][["subject_identifier", "visit_code", "direction", "complications_count"]]


In [ ]:
df_subject_transfer = read_frame_edc(SubjectTransfer.objects.all())
df_subject_transfer = df_subject_transfer.merge(df_main[["subject_identifier", "assignment", "primary_cohort", "primary_cohort_str", "ncd", "hiv_only"]], on="subject_identifier", how="left")


In [ ]:

def get_need_specialized_treatment(s):
    if "need_specialized_treatment" in s["transfer_reason"]:
        return YES
    return NO

def get_need_specialized_treatment_from_other(s):
    responses = ["Low viramia", "High viral load", "Had plumonary tuberculosis"]
    if s["need_specialized_treatment"] !=YES:
        if s["transfer_reason"] in responses:
            return YES
    return s["need_specialized_treatment"]

df_subject_transfer["need_specialized_treatment"] = df_subject_transfer.apply(get_need_specialized_treatment, axis=1)
df_subject_transfer["need_specialized_treatment"] = df_subject_transfer.apply(get_need_specialized_treatment_from_other, axis=1)

In [ ]:
df = df_main.copy()
df = df.merge(df_subject_transfer[["subject_identifier", "need_specialized_treatment"]], on="subject_identifier", how="left")
df.loc[df.need_specialized_treatment.isna(), "need_specialized_treatment"] = NO
tbl_dct = get_primary_cohorts_by_categorical_column(
    df, "need_specialized_treatment")
dftbl = pd.DataFrame(tbl_dct)
dftbl.loc[dftbl.Statistics==YES, "Statistics"] = "Facility to hospital/tertiary care"
dftbl

# df_subject_transfer

In [ ]:

careseeka_df = get_crf("intecomm_subject.careseekinga", subject_visit_model="intecomm_subject.subjectvisit")
careseeka_df = careseeka_df.merge(df_main[["subject_identifier", "assignment", "primary_cohort", "primary_cohort_str", "ncd", "hiv_only"]], on=["subject_identifier"], how="left")
careseeka_df["care_visit_minutes"] = careseeka_df.care_visit_tdelta.dt.total_seconds()/60

tbl_dct = get_primary_cohorts_for_continuous_var(
    careseeka_df, "care_visit_minutes", statistics=["median"])
dftbl = pd.DataFrame(tbl_dct)
dftbl


In [ ]:
careseeka_df = get_crf("intecomm_subject.careseekinga", subject_visit_model="intecomm_subject.subjectvisit")
careseeka_df = careseeka_df.merge(df_main[["subject_identifier", "assignment", "primary_cohort", "primary_cohort_str", "ncd", "hiv_only"]], on=["subject_identifier"], how="left")
careseeka_df["travel_minutes"] = careseeka_df.travel_tdelta.dt.total_seconds()/60

tbl_dct = get_primary_cohorts_for_continuous_var(
    careseeka_df, "travel_minutes", statistics=["median"])
dftbl = pd.DataFrame(tbl_dct)
dftbl


In [ ]:
careseeka_df.groupby(by=["appt_type", "assignment"]).size()

In [ ]:
df_appointment[df_appointment.appt_status.isin([INCOMPLETE_APPT, IN_PROGRESS_APPT, COMPLETE_APPT])].groupby(by=["assignment", "subject_identifier"]).size().to_frame("appointments").groupby(by=["assignment"]).appointments.describe()

Compare mean proportions of missed visits group A to group B

Group A is less likely to miss a visit than group B (t=3.548 p=0.0004).
The mean proportion of missed visits in group A is lower than in group B.

In [ ]:

df_tmp = pd.merge(df_appointment, df_main[["subject_identifier", "assignment"]], how="left", on="subject_identifier")
df_tmp = df_tmp[(df_tmp.visit_code_sequence==0) &
       ~(df_tmp.appt_status==SKIPPED_APPT) &
       ~(df_tmp.appt_status==NEW_APPT)
]
df_tmp = df_tmp.groupby(by=["subject_identifier", "appt_timing", "assignment"]).size().to_frame().reset_index()
df_tmp = df_tmp.pivot_table(index=["subject_identifier","assignment"], columns=["appt_timing"], values=0).fillna(0).astype(int).reset_index()
df_tmp["total_appts"] = df_tmp["missed"] + df_tmp["ontime"]
df_tmp["prop_missed"] = df_tmp["missed"] / df_tmp["total_appts"]


t_stat, p_value = ttest_ind(df_tmp[df_tmp.assignment=="a"]["prop_missed"], df_tmp[df_tmp.assignment=="b"]["prop_missed"], equal_var=False)
t_stat, p_value


In [ ]:
from edc_appointment.constants import (
    SKIPPED_APPT, NEW_APPT, COMPLETE_APPT, IN_PROGRESS_APPT, INCOMPLETE_APPT
)

def attended(row):
    if row.appt_status in [COMPLETE_APPT, IN_PROGRESS_APPT, INCOMPLETE_APPT]:
        return 1
    else:
        return 0

df_main = get_df_main_1858(None)
df_appointment = get_appointments()
df_tmp = pd.merge(df_appointment, df_main[["subject_identifier", "assignment"]], how="left", on="subject_identifier")
df_tmp = df_tmp[(df_tmp.visit_code_sequence==0) &
       ~(df_tmp.appt_status==SKIPPED_APPT) &
       ~(df_tmp.appt_status==NEW_APPT)
]
df_tmp = df_tmp.groupby(by=["subject_identifier", "appt_timing", "assignment"]).size().to_frame().reset_index()
df_tmp = df_tmp.pivot_table(index=["subject_identifier","assignment"], columns=["appt_timing"], values=0).fillna(0).astype(int).reset_index()
df_tmp = df_tmp.rename(columns={"missed": "missed_appts", "ontime": "attended_appts"})
df_tmp["total_appts"] = df_tmp["missed_appts"] + df_tmp["attended_appts"]
df_tmp["prop_appts_missed"] = df_tmp["missed_appts"] / df_tmp["total_appts"]

df_main = df_main.merge(df_tmp[["subject_identifier", "missed_appts", "attended_appts", "total_appts", "prop_appts_missed"]], how="left", on="subject_identifier")

df_visit = get_subject_visit(model="intecomm_subject.subjectvisit")
df_visit = df_visit[(df_visit.visit_code_sequence==0) & (df_visit.reason!="missed")].groupby(by=["subject_identifier", "baseline_datetime", "last_visit_datetime"]).size().to_frame().reset_index()
df_visit.columns = ["subject_identifier", "baseline_datetime", "last_visit_datetime", "attended_visits"]
df_main = df_main.merge(df_visit[["subject_identifier", "baseline_datetime", "last_visit_datetime"]], how="left", on="subject_identifier")
df_main


In [ ]:
df_main["duration"] = df_main["last_visit_datetime"] - df_main["baseline_datetime"]
df_main["duration"] = df_main["duration"].dt.days
df_main["duration"].mean()

In [ ]:
df_main["in_care_365"] = df_main["duration"] > 182


In [ ]:

df_odds = df_main.groupby(by=["assignment", "in_care_365"]).size().to_frame().reset_index()
df_odds.columns = ["assignment", "in_care_365", "count"]
df_odds['in_care_365'] = df_odds["in_care_365"].apply(lambda x: YES if x is True else NO)
df_odds

In [ ]:
import scipy.stats as stats

df_odds = df_odds.set_index("assignment")
# df_odds
df_odds.groupby(level="assignment").sum().values[1]